# **Importing necessary Libraries**

In [ ]:
!pip install river

# **Loading the dataset**

In [ ]:
import pandas as pd

# Load the transformed dataset
df = pd.read_csv('/content/transformed_energy_data_ALB0331.csv')

# Display the first few rows of the dataset to understand its structure
print(df.head())

# **Understanding the Dataset**

In [ ]:
# Basic info about the dataset
print("Dataset Info:")
df.info()

# Basic statistics (for numerical columns)
print("\nSummary Statistics:")
print(df.describe())

# Exploratory Data Analysis:
# Importing Visualization Libraries , Setting Plot Style and Visualizing Numerical Data Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set(style="whitegrid")

# Visualize the distribution of numerical columns
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

# Plot histograms for all numerical features
df[numerical_columns].hist(figsize=(14, 10), bins=30, color='skyblue', edgecolor='black')
plt.tight_layout()
plt.show()

# Box plots for the distribution of values (checking for outliers)
for col in numerical_columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[col], color='lightgreen')
    plt.title(f'Box plot of {col}')
    plt.show()

# **Visualizing Relationships Between Features using Pair Plot and Correlation Heatmap**



In [ ]:
# Pair plot to visualize relationships between features
sns.pairplot(df[numerical_columns], height=2.5)
plt.show()

# Calculate the correlation matrix
corr = df[numerical_columns].corr()

# Heatmap of correlations
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()


# **Creating a Scatter Plot to Visualize the Relationship Between Avg$PerMWHr and Med$PerMWHr**

In [ ]:
# Scatter plot for Avg$PerMWHr vs. Med$PerMWHr
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df['Avg$PerMWHr'], y=df['Med$PerMWHr'], hue=df['Year'], palette='viridis')
plt.title('Avg$PerMWHr vs. Med$PerMWHr')
plt.xlabel('Avg$PerMWHr')
plt.ylabel('Med$PerMWHr')
plt.show()

# **Plotting Average Price Per Megawatt-hour Over Time and Creating the Line Plot**

In [ ]:
# Plot Avg$PerMWHr over time
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Date', y='Avg$PerMWHr', hue='TradingPeriod', palette='coolwarm', lw=2)
plt.title('Avg$PerMWHr Over Time')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.legend(title='Trading Period', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


# **Calculating Monthly Average and Plotting the Trend**

In [ ]:
# Calculate average of Avg$PerMWHr per month
monthly_avg = df.groupby(['Year', 'Month'])['Avg$PerMWHr'].mean().reset_index()

# Plot Monthly Avg$PerMWHr trends
plt.figure(figsize=(10, 6))
sns.lineplot(data=monthly_avg, x='Month', y='Avg$PerMWHr', hue='Year', marker='o')
plt.title('Monthly Avg$PerMWHr Trends')
plt.xlabel('Month')
plt.ylabel('Avg$PerMWHr')
plt.show()

# **Checking for Sinusoidal Trends in Trading Periods**

In [ ]:
# Check if there's any sinusoidal trend for TradingPeriod (periodicity)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='TradingPeriod', y='Avg$PerMWHr', hue='Month', palette='Set2')
plt.title('Avg$PerMWHr Across Trading Periods')
plt.xlabel('Trading Period')
plt.ylabel('Avg$PerMWHr')
plt.show()


# **Correlation Heatmap for key features**

In [ ]:
# Correlation plot for key features (e.g., Avg$PerMWHr, Med$PerMWHr, SinPeriod, etc.)
plt.figure(figsize=(12, 8))
sns.heatmap(df[numerical_columns].corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()


# **Anomaly Detection :**
# Importing Libraries and Loading and Preparing the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Load the dataset
df = pd.read_csv('/content/transformed_energy_data_ALB0331.csv')

# Convert Date components to datetime
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])

# Drop non-numeric columns
df_numeric = df.select_dtypes(include=['float64', 'int64']).copy()

# Display first few rows
print(df_numeric.head())

# **Anomaly Detection with Isolation Forest**

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# Fit the model
df_numeric['Anomaly_Score'] = iso_forest.fit_predict(df_numeric)

# Mark anomalies (Anomaly_Score = -1 means an anomaly)
df_numeric['Anomaly'] = df_numeric['Anomaly_Score'].apply(lambda x: 'Anomaly' if x == -1 else 'Normal')

# Display anomaly counts
print(df_numeric['Anomaly'].value_counts())

# Merge back with original dataset
df['Anomaly'] = df_numeric['Anomaly']

# **Visualizing Anomalies in Average Price Over Time**

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='Date', y='Avg$PerMWHr', hue='Anomaly', palette={'Normal': 'blue', 'Anomaly': 'red'}, alpha=0.7)
plt.title('Anomaly Detection in Avg$PerMWHr Over Time')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.legend(title='Anomaly Status')
plt.show()


# **Creating a Box Plot to Visualize the Distribution of Avg$PerMWHr for Normal Data Points and Anomalies**

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Anomaly', y='Avg$PerMWHr', palette={'Normal': 'lightblue', 'Anomaly': 'red'})
plt.title('Distribution of Avg$PerMWHr for Normal vs Anomalies')
plt.show()

# **Creating a Box Plot to Visualize Anomalies Across Trading Periods**

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='TradingPeriod', y='Avg$PerMWHr', hue='Anomaly', palette={'Normal': 'lightblue', 'Anomaly': 'red'})
plt.title('Avg$PerMWHr Across Trading Periods (Anomalies Highlighted)')
plt.xticks(rotation=45)
plt.show()


# **Concept Drift Detection with ADWIN : Saving and Visualizing Drift Points**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from river.drift import ADWIN

# Step 1: Load dataset
file_path = "/content/transformed_energy_data_ALB0331.csv"
df = pd.read_csv(file_path)

# Step 2: Sort data by time (ensuring correct order)
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
df = df.sort_values(by=['Date', 'TradingPeriod'])

# Step 3: Initialize ADWIN detector
adwin = ADWIN()

# Step 4: Monitor for drift in the target variable (Avg$PerMWHr)
drift_points = []
mean_values = []  # To track mean over time

for i, value in enumerate(df['Avg$PerMWHr']):
    adwin.update(value)
    mean_values.append(np.mean(df['Avg$PerMWHr'][:i+1]))  # Compute mean manually
    if adwin.drift_detected:
        drift_points.append(i)  # Store drift index
        print(f"Concept drift detected at index {i}, Date: {df.iloc[i]['Date']}")

# Step 5: Save the concept drift points to a separate file
drift_df = df.iloc[drift_points]  # Get rows where drift was detected
drift_df.to_csv('concept_drift_points.csv', index=False)

# Step 6: Visualize detected drifts
plt.figure(figsize=(12,6))
plt.plot(df.index, df['Avg$PerMWHr'], label="Avg$PerMWHr", alpha=0.7)
plt.scatter(df.index[drift_points], df['Avg$PerMWHr'].iloc[drift_points], color='red', label="Drift Detected", marker='x')
plt.xlabel("Time")
plt.ylabel("Avg$PerMWHr")
plt.title("Concept Drift Detection using ADWIN")
plt.legend()
plt.show()


# **Concept Drift Detection using SARIMA and ADWIN**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from river.drift import ADWIN
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Step 1: Load dataset
file_path = "/content/transformed_energy_data_ALB0331.csv"
df = pd.read_csv(file_path)

# Step 2: Sort data by time (ensuring correct order)
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
df = df.sort_values(by=['Date', 'TradingPeriod'])

# Step 3: Fit the SARIMA model to account for seasonality and trend
sarima_model = SARIMAX(df['Avg$PerMWHr'],
                       order=(1, 1, 1),  # ARIMA order
                       seasonal_order=(1, 1, 1, 12))  # Seasonal order with 12 months seasonality
results = sarima_model.fit()

# Step 4: Calculate the residuals (errors between actual values and predicted values)
df['SARIMA_Residuals'] = df['Avg$PerMWHr'] - results.fittedvalues

# Step 5: Initialize the ADWIN detector for concept drift detection
adwin = ADWIN()

# Step 6: Monitor for drift in the SARIMA residuals
drift_points = []
mean_values = []  # To track mean over time

for i, residual in enumerate(df['SARIMA_Residuals']):
    adwin.update(residual)
    mean_values.append(np.mean(df['SARIMA_Residuals'][:i+1]))  # Compute mean manually
    if adwin.drift_detected:
        drift_points.append(i)  # Store drift index
        print(f"Concept drift detected at index {i}, Date: {df.iloc[i]['Date']}")

# Step 7: Save the concept drift points to a separate file
drift_df = df.iloc[drift_points]  # Get rows where drift was detected
drift_df.to_csv('concept_drift_points.csv', index=False)

# Step 8: Visualize detected drifts
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['SARIMA_Residuals'], label="SARIMA Residuals", alpha=0.7)
plt.scatter(df.index[drift_points], df['SARIMA_Residuals'].iloc[drift_points], color='red', label="Drift Detected", marker='x')
plt.xlabel("Time")
plt.ylabel("SARIMA Residuals")
plt.title("Concept Drift Detection using SARIMA Residuals and ADWIN")
plt.legend()
plt.show()

# **Data Modelling : Building a Predictive Model using LightGBM and SARIMA with Concept Drift Detection**

# **Step 1: Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from river.drift import ADWIN
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib

# **Step 2: Loading and Preprocessing Data**

In [ ]:
# Step 1: Load dataset
file_path = "/content/transformed_energy_data_ALB0331.csv"
df = pd.read_csv(file_path)

# Step 2: Define features and target
features = ['Year', 'Month', 'Day', 'TradingPeriod', 'PrevAvg$PerMWHr', 'PrevMed$PerMWHr', 'SinPeriod', 'CosPeriod', 'SinDate', 'CosDate']
target = 'Avg$PerMWHr'

# Step 3: Add Date column and sort data
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
df = df.sort_values(by=['Date', 'TradingPeriod'])

# **Step 3: Modeling with SARIMA and Feature Engineering**

In [ ]:
# Step 4: Fit SARIMA model to capture seasonal and trend components
sarima_model = SARIMAX(df['Avg$PerMWHr'],
                       order=(1, 1, 1),  # ARIMA order
                       seasonal_order=(1, 1, 1, 12))  # Seasonal order with 12 months seasonality
sarima_results = sarima_model.fit()

# Step 5: Calculate SARIMA residuals and add them as a feature
df['SARIMA_Residuals'] = df['Avg$PerMWHr'] - sarima_results.fittedvalues

# Step 6: Prepare features and target with the new residuals feature
df['SARIMA_Residuals'] = df['SARIMA_Residuals'].fillna(0)  # Fill NaN residuals
features.append('SARIMA_Residuals')  # Add residuals as a feature

# **Step 4: Data Splitting and Preprocessing**

In [ ]:
# Step 7: Split the data for training and testing
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Handle missing values
X_train.fillna(X_train.mean(), inplace=True)
X_test.fillna(X_train.mean(), inplace=True)

# Step 9: Outlier removal
Q1_train = X_train.quantile(0.25)
Q3_train = X_train.quantile(0.75)
IQR_train = Q3_train - Q1_train

X_train_outliers_removed = X_train[~((X_train < (Q1_train - 1.5 * IQR_train)) | (X_train > (Q3_train + 1.5 * IQR_train))).any(axis=1)]
y_train_outliers_removed = y_train[X_train_outliers_removed.index]
X_test_outliers_removed = X_test[~((X_test < (Q1_train - 1.5 * IQR_train)) | (X_test > (Q3_train + 1.5 * IQR_train))).any(axis=1)]
y_test_outliers_removed = y_test[X_test_outliers_removed.index]

# **Step 5: Model Training and Concept Drift Detection**

In [ ]:
# Step 10: Initialize ADWIN for concept drift detection
drift_detector = ADWIN()

# Step 11: Train LightGBM model
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=5)
lgb_model.fit(X_train_outliers_removed, y_train_outliers_removed)

# Step 12: Track metrics before and after concept drift integration
metrics_before = {}
metrics_after = {}

# Step 13: Predictions before concept drift adaptation (without outlier removal)
lgb_preds_before = lgb_model.predict(X_test_outliers_removed)
rmse_before = np.sqrt(mean_squared_error(y_test_outliers_removed, lgb_preds_before))
mape_before = mean_absolute_percentage_error(y_test_outliers_removed, lgb_preds_before)
accuracy_before = 1 - mape_before
metrics_before['RMSE'] = rmse_before
metrics_before['MAPE'] = mape_before
metrics_before['Accuracy'] = accuracy_before

# Predictions without removing outliers (for the original test set)
lgb_preds_without_outliers = lgb_model.predict(X_test)
rmse_without_outliers = np.sqrt(mean_squared_error(y_test, lgb_preds_without_outliers))
mape_without_outliers = mean_absolute_percentage_error(y_test, lgb_preds_without_outliers)
accuracy_without_outliers = 1 - mape_without_outliers

# Step 14: Monitor drift in the SARIMA residuals
for i in range(len(X_test_outliers_removed)):
    prediction = lgb_preds_before[i]
    residual = df['SARIMA_Residuals'].iloc[i]  # Use SARIMA residuals for drift detection
    drift_detector.update(residual)

    if drift_detector.drift_detected:
        print(f"Drift detected at index {i}. Retraining model...")

        # Retrain LightGBM model
        lgb_model.fit(X_train_outliers_removed, y_train_outliers_removed)

# Step 15: Predictions after concept drift integration
lgb_preds_after = lgb_model.predict(X_test_outliers_removed)
rmse_after = np.sqrt(mean_squared_error(y_test_outliers_removed, lgb_preds_after))
mape_after = mean_absolute_percentage_error(y_test_outliers_removed, lgb_preds_after)
accuracy_after = 1 - mape_after
metrics_after['RMSE'] = rmse_after
metrics_after['MAPE'] = mape_after
metrics_after['Accuracy'] = accuracy_after

# **Step 6: Model Evaluation and Saving**

In [ ]:
# Step 16: Save the trained model for future use
joblib.dump(lgb_model, 'trained_lgb_model_with_concept_drift.joblib')
print("Model saved to 'trained_lgb_model_with_concept_drift.joblib'")

# Step 17: Save predictions vs actual values to a CSV file
predicted_vs_actual = pd.DataFrame({
    'Actual': y_test_outliers_removed,
    'Predicted': lgb_preds_after
})
predicted_vs_actual.to_csv("predicted_vs_actual_test_with_drift.csv", index=False)

# Step 18: Evaluate the model after concept drift detection
print(f"LightGBM Model (After Concept Drift) - RMSE: {metrics_after['RMSE']}")
print(f"LightGBM Model (After Concept Drift) - MAPE: {metrics_after['MAPE']}")
print(f"LightGBM Model (After Concept Drift) - Accuracy: {metrics_after['Accuracy']}")

# Step 19: Evaluate the model without removing outliers
print(f"LightGBM Model (Without Outlier Removal) - RMSE: {rmse_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - MAPE: {mape_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - Accuracy: {accuracy_without_outliers}")

# **Step 7: Visualization**

In [ ]:
# Step 20: Plot LightGBM Feature Importance
lgb.plot_importance(lgb_model, max_num_features=10, importance_type='split', title="LightGBM Feature Importance")
plt.show()

# Step 21: Plot Predicted vs Actual (With and Without Outlier Removal)

plt.figure(figsize=(12, 6))

# Plot with outlier removal
plt.subplot(1, 2, 1)
# Sort by Date and interpolate actual and predicted values for smooth continuous curves
sorted_test_data = df.iloc[X_test_outliers_removed.index].sort_values(by='Date')

plt.plot(sorted_test_data['Date'], sorted_test_data['Avg$PerMWHr'], label='Actual Values', color='blue')
plt.plot(sorted_test_data['Date'], lgb_preds_after, label='Predicted Values (After Drift)', color='red', linestyle='--')
plt.title('Predicted vs Actual (Outliers Removed)')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.xticks(rotation=45)
plt.legend()

# Plot without outlier removal
plt.subplot(1, 2, 2)
# Sort by Date and interpolate actual and predicted values for continuous curves (no outlier removal)
sorted_test_data_no_outliers = df.iloc[X_test.index].sort_values(by='Date')

plt.plot(sorted_test_data_no_outliers['Date'], sorted_test_data_no_outliers['Avg$PerMWHr'], label='Actual Values', color='blue')
plt.plot(sorted_test_data_no_outliers['Date'], lgb_preds_without_outliers, label='Predicted Values (No Outlier Removal)', color='red', linestyle='--')
plt.title('Predicted vs Actual (No Outliers Removed)')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

# Step 22: Plot Seasonal Trend for Each Month (Continuous Line Plot)
monthly_avg = df.groupby(df['Date'].dt.month)['Avg$PerMWHr'].mean()
plt.figure(figsize=(10, 6))
plt.plot(monthly_avg.index, monthly_avg.values, color='skyblue', marker='o', linestyle='-', linewidth=2)
plt.title('Seasonal Trend by Month')
plt.xlabel('Month')
plt.ylabel('Avg$PerMWHr')
plt.xticks(monthly_avg.index)
plt.tight_layout()
plt.show()

# Step 23: Plot Seasonal Trend for Each Day (Continuous Line Plot)
daily_avg = df.groupby(df['Date'].dt.day)['Avg$PerMWHr'].mean()
plt.figure(figsize=(10, 6))
plt.plot(daily_avg.index, daily_avg.values, color='lightcoral', marker='o', linestyle='-', linewidth=2)
plt.title('Seasonal Trend by Day')
plt.xlabel('Day')
plt.ylabel('Avg$PerMWHr')
plt.xticks(daily_avg.index)
plt.tight_layout()
plt.show()

# **Data modelling : full combined code with results **

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from river.drift import ADWIN
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib

# Step 1: Load dataset
file_path = "/content/transformed_energy_data_ALB0331.csv"
df = pd.read_csv(file_path)

# Step 2: Define features and target
features = ['Year', 'Month', 'Day', 'TradingPeriod', 'PrevAvg$PerMWHr', 'PrevMed$PerMWHr', 'SinPeriod', 'CosPeriod', 'SinDate', 'CosDate']
target = 'Avg$PerMWHr'

# Step 3: Add Date column and sort data
df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
df = df.sort_values(by=['Date', 'TradingPeriod'])

# Step 4: Fit SARIMA model to capture seasonal and trend components
sarima_model = SARIMAX(df['Avg$PerMWHr'],
                       order=(1, 1, 1),  # ARIMA order
                       seasonal_order=(1, 1, 1, 12))  # Seasonal order with 12 months seasonality
sarima_results = sarima_model.fit()

# Step 5: Calculate SARIMA residuals and add them as a feature
df['SARIMA_Residuals'] = df['Avg$PerMWHr'] - sarima_results.fittedvalues

# Step 6: Prepare features and target with the new residuals feature
df['SARIMA_Residuals'] = df['SARIMA_Residuals'].fillna(0)  # Fill NaN residuals
features.append('SARIMA_Residuals')  # Add residuals as a feature

# Step 7: Split the data for training and testing
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Handle missing values
X_train.fillna(X_train.mean(), inplace=True)
X_test.fillna(X_train.mean(), inplace=True)

# Step 9: Outlier removal
Q1_train = X_train.quantile(0.25)
Q3_train = X_train.quantile(0.75)
IQR_train = Q3_train - Q1_train

X_train_outliers_removed = X_train[~((X_train < (Q1_train - 1.5 * IQR_train)) | (X_train > (Q3_train + 1.5 * IQR_train))).any(axis=1)]
y_train_outliers_removed = y_train[X_train_outliers_removed.index]
X_test_outliers_removed = X_test[~((X_test < (Q1_train - 1.5 * IQR_train)) | (X_test > (Q3_train + 1.5 * IQR_train))).any(axis=1)]
y_test_outliers_removed = y_test[X_test_outliers_removed.index]

# Step 10: Initialize ADWIN for concept drift detection
drift_detector = ADWIN()

# Step 11: Train LightGBM model
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=5)
lgb_model.fit(X_train_outliers_removed, y_train_outliers_removed)

# Step 12: Track metrics before and after concept drift integration
metrics_before = {}
metrics_after = {}

# Step 13: Predictions before concept drift adaptation (without outlier removal)
lgb_preds_before = lgb_model.predict(X_test_outliers_removed)
rmse_before = np.sqrt(mean_squared_error(y_test_outliers_removed, lgb_preds_before))
mape_before = mean_absolute_percentage_error(y_test_outliers_removed, lgb_preds_before)
accuracy_before = 1 - mape_before
metrics_before['RMSE'] = rmse_before
metrics_before['MAPE'] = mape_before
metrics_before['Accuracy'] = accuracy_before

# Predictions without removing outliers (for the original test set)
lgb_preds_without_outliers = lgb_model.predict(X_test)
rmse_without_outliers = np.sqrt(mean_squared_error(y_test, lgb_preds_without_outliers))
mape_without_outliers = mean_absolute_percentage_error(y_test, lgb_preds_without_outliers)
accuracy_without_outliers = 1 - mape_without_outliers

# Step 14: Monitor drift in the SARIMA residuals (used as a proxy for drift in LightGBM predictions)
for i in range(len(X_test_outliers_removed)):
    prediction = lgb_preds_before[i]
    residual = df['SARIMA_Residuals'].iloc[i]  # Use SARIMA residuals for drift detection
    drift_detector.update(residual)

    if drift_detector.drift_detected:
        print(f"Drift detected at index {i}. Retraining model...")

        # Retrain LightGBM model
        lgb_model.fit(X_train_outliers_removed, y_train_outliers_removed)

# Step 15: Predictions after concept drift integration
lgb_preds_after = lgb_model.predict(X_test_outliers_removed)
rmse_after = np.sqrt(mean_squared_error(y_test_outliers_removed, lgb_preds_after))
mape_after = mean_absolute_percentage_error(y_test_outliers_removed, lgb_preds_after)
accuracy_after = 1 - mape_after
metrics_after['RMSE'] = rmse_after
metrics_after['MAPE'] = mape_after
metrics_after['Accuracy'] = accuracy_after

# Step 16: Save the trained model for future use
joblib.dump(lgb_model, 'trained_lgb_model_with_concept_drift.joblib')
print("Model saved to 'trained_lgb_model_with_concept_drift.joblib'")

# Step 17: Save predictions vs actual values to a CSV file
predicted_vs_actual = pd.DataFrame({
    'Actual': y_test_outliers_removed,
    'Predicted': lgb_preds_after
})
predicted_vs_actual.to_csv("predicted_vs_actual_test_with_drift.csv", index=False)

# Step 18: Evaluate the model after concept drift detection
print(f"LightGBM Model (After Concept Drift) - RMSE: {metrics_after['RMSE']}")
print(f"LightGBM Model (After Concept Drift) - MAPE: {metrics_after['MAPE']}")
print(f"LightGBM Model (After Concept Drift) - Accuracy: {metrics_after['Accuracy']}")

# Step 19: Evaluate the model without removing outliers
print(f"LightGBM Model (Without Outlier Removal) - RMSE: {rmse_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - MAPE: {mape_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - Accuracy: {accuracy_without_outliers}")

# Step 20: Plot LightGBM Feature Importance
lgb.plot_importance(lgb_model, max_num_features=10, importance_type='split', title="LightGBM Feature Importance")
plt.show()

# Step 21: Plot Predicted vs Actual (With and Without Outlier Removal)

plt.figure(figsize=(12, 6))

# Plot with outlier removal
plt.subplot(1, 2, 1)
# Sort by Date and interpolate actual and predicted values for smooth continuous curves
sorted_test_data = df.iloc[X_test_outliers_removed.index].sort_values(by='Date')

plt.plot(sorted_test_data['Date'], sorted_test_data['Avg$PerMWHr'], label='Actual Values', color='blue')
plt.plot(sorted_test_data['Date'], lgb_preds_after, label='Predicted Values (After Drift)', color='red', linestyle='--')
plt.title('Predicted vs Actual (Outliers Removed)')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.xticks(rotation=45)
plt.legend()

# Plot without outlier removal
plt.subplot(1, 2, 2)
# Sort by Date and interpolate actual and predicted values for continuous curves (no outlier removal)
sorted_test_data_no_outliers = df.iloc[X_test.index].sort_values(by='Date')

plt.plot(sorted_test_data_no_outliers['Date'], sorted_test_data_no_outliers['Avg$PerMWHr'], label='Actual Values', color='blue')
plt.plot(sorted_test_data_no_outliers['Date'], lgb_preds_without_outliers, label='Predicted Values (No Outlier Removal)', color='red', linestyle='--')
plt.title('Predicted vs Actual (No Outliers Removed)')
plt.xlabel('Date')
plt.ylabel('Avg$PerMWHr')
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

# Step 22: Plot Seasonal Trend for Each Month (Continuous Line Plot)
monthly_avg = df.groupby(df['Date'].dt.month)['Avg$PerMWHr'].mean()
plt.figure(figsize=(10, 6))
plt.plot(monthly_avg.index, monthly_avg.values, color='skyblue', marker='o', linestyle='-', linewidth=2)
plt.title('Seasonal Trend by Month')
plt.xlabel('Month')
plt.ylabel('Avg$PerMWHr')
plt.xticks(monthly_avg.index)
plt.tight_layout()
plt.show()

# Step 23: Plot Seasonal Trend for Each Day (Continuous Line Plot)
daily_avg = df.groupby(df['Date'].dt.day)['Avg$PerMWHr'].mean()
plt.figure(figsize=(10, 6))
plt.plot(daily_avg.index, daily_avg.values, color='lightcoral', marker='o', linestyle='-', linewidth=2)
plt.title('Seasonal Trend by Day')
plt.xlabel('Day')
plt.ylabel('Avg$PerMWHr')
plt.xticks(daily_avg.index)
plt.tight_layout()
plt.show()

# **Calculating and Visualizing Model Performance Metrics with and without outlier removal**

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

# Compute additional metrics
r2_after = r2_score(y_test_outliers_removed, lgb_preds_after)
r2_without_outliers = r2_score(y_test, lgb_preds_without_outliers)

mae_after = mean_absolute_error(y_test_outliers_removed, lgb_preds_after)
mae_without_outliers = mean_absolute_error(y_test, lgb_preds_without_outliers)

smape_after = smape(y_test_outliers_removed, lgb_preds_after)
smape_without_outliers = smape(y_test, lgb_preds_without_outliers)

# Print additional metrics
print(f"LightGBM Model (After Concept Drift) - R2 Score: {r2_after}")
print(f"LightGBM Model (After Concept Drift) - MAE: {mae_after}")
print(f"LightGBM Model (After Concept Drift) - SMAPE: {smape_after}")

print(f"LightGBM Model (Without Outlier Removal) - R2 Score: {r2_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - MAE: {mae_without_outliers}")
print(f"LightGBM Model (Without Outlier Removal) - SMAPE: {smape_without_outliers}")

# Plot all accuracy metrics
metrics_labels = ['RMSE', 'MAPE', 'Accuracy', 'R2 Score', 'MAE', 'SMAPE']
metrics_after_values = [rmse_after, mape_after, accuracy_after, r2_after, mae_after, smape_after]
metrics_without_outliers_values = [rmse_without_outliers, mape_without_outliers, accuracy_without_outliers, r2_without_outliers, mae_without_outliers, smape_without_outliers]

x = np.arange(len(metrics_labels))
width = 0.35  # Bar width

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, metrics_after_values, width, label='After Concept Drift')
rects2 = ax.bar(x + width/2, metrics_without_outliers_values, width, label='Without Outlier Removal')

ax.set_ylabel('Metric Values')
ax.set_title('Comparison of Model Accuracy Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, rotation=45)
ax.legend()

plt.show()


# **Detecting and Saving Outliers in a seperate file**

In [ ]:
# Step: Detect Outliers
outliers_train = X_train[((X_train < (Q1_train - 1.5 * IQR_train)) | (X_train > (Q3_train + 1.5 * IQR_train))).any(axis=1)]
outliers_test = X_test[((X_test < (Q1_train - 1.5 * IQR_train)) | (X_test > (Q3_train + 1.5 * IQR_train))).any(axis=1)]

# Combine outliers from train and test sets
outliers_train['Target'] = y_train[outliers_train.index]
outliers_test['Target'] = y_test[outliers_test.index]
outliers_combined = pd.concat([outliers_train, outliers_test])

# Save outliers to a CSV file
outliers_combined.to_csv("detected_outliers.csv", index=False)
print("Outliers saved to 'detected_outliers.csv'")
